In [ ]:
# Gem data
import gemlog
from pathlib import Path
GEM_DATA_PATH = Path('/Volumes') / 'tachyon' / '20260403_DOWNLOAD'  / 'gem_data' / 'mseed'
from obspy import read, UTCDateTime, Stream
launchtime = UTCDateTime('2026-04-01T22:35:12')
duration = 900
pretrigger = 1800
posttrigger = pretrigger
import glob

def fix_gem_trace_id(tr):
    id_dict = {
        '273': '1R.B24LT..HDF',
        '287': '1R.B24RE..HDF',
        '288': '1R.B24..HDF',
        '275': '1R.B24..HDI',
        '296': '1R.B29..HDF',
        '292': '1R.B29..HDI',
        '369': '1R.B12..HDF',
        '290': '1R.B23..HDF',
        '256': '1R.B03..HDF',
        '254': '1R.B03..HDI',
        '246': '1R.B14..HDF',
        '244': '1R.B14..HDI',
        '362': '1R.B01..HDF',
        '251': '1R.B01..HDI',
        '274': '1R.B07.10.HDF',
        '365': '1R.B07..HDF',
        '285': '1R.B20..HDF',
        '361': '1R.B20..HDI',
        '294': '1R.B20.10.HDI',
        '364': '1R.B20W..HDF',
        '366': '1R.B20S..HDF',
    }
    tr.id = id_dict.get(tr.stats.station, tr.id)


files = glob.glob(str(GEM_DATA_PATH / '2026-04-01*.mseed'))
print(f'Found {len(files)} files')
# Read the first file to get the start time
master_stream = Stream()
for file in files:
    print(f'Reading file: {file}')  
    st = read(file).trim(starttime=launchtime - pretrigger, endtime=launchtime + duration + posttrigger)
    if len(st) > 0:
        print(f'Found data in file: {file}')
        ## if you used a config file to set the Gem's gain to low, change the gain setting below
        st2 = gemlog.deconvolve_gem_response(st, gain='high') 
        ## filter data above 1 Hz (lower frequencies are often wind noise)
        st2.filter("highpass", freq=0.5)
        for tr in st2:
            ## deconvolve the instrument response
            tr.trim(starttime=launchtime - duration, endtime=launchtime + duration)
            fix_gem_trace_id(tr)
            master_stream.append(tr)
master_stream.plot(equal_scale=False, outfile='artemis2_gem_launch.png');#, size=(800, 600), title='GEM Data around Launch Time');
master_stream.write('artemis2_gem_launch.mseed', format='MSEED')

In [ ]:
master_stream_zoom =master_stream.copy()
master_stream_zoom.trim(starttime=launchtime - 60, endtime=launchtime + 240)
master_stream_zoom.plot(equal_scale=False, outfile='artemis2_gem_launch_zoom.png');

In [ ]:
import numpy as np
peak_amplitudes = {}
for tr in master_stream_zoom.copy():
    tr.detrend(type='demean')
    data = np.abs(tr.data)
    peak_amplitudes[tr.id] = data.max()
peak_amplitudes = {k: v for k, v in sorted(peak_amplitudes.items(), key=lambda item: item[1], reverse=True)}
print('Peak amplitudes for each trace (sorted):')

import pandas as pd

df = pd.DataFrame(list(peak_amplitudes.items()), columns=['Trace ID', 'Peak Amplitude (Pa)'])
df.to_csv('gem_peak_amplitudes.csv', index=False)
df['Peak Amplitude (Pa)'] = df['Peak Amplitude (Pa)'].round(1)
display(df)
    

In [ ]:
# Plot spectrograms for each trace
for tr in master_stream_zoom.copy():
    tr.detrend(type='demean')
    tr.spectrogram(log=False, title=f'Spectrogram for {tr.id}', dbscale=True)#, outfile=f'spectrogram_{tr.id}.png')

In [ ]:
from obspy import read
import numpy as np
import matplotlib.pyplot as plt

def plot_spectrum(tr):
    # Detrend and demean the trace
    tr.detrend(type='demean')
    
    # Apply a taper to minimize edge effects
    tr.taper(max_percentage=0.05)

    # sampling interval
    dt = tr.stats.delta

    # FFT
    n = tr.stats.npts
    freq = np.fft.rfftfreq(n, d=dt)
    amp = 2.0 / n * np.abs(np.fft.rfft(tr.data))
    amp[0] /= 2.0  # keep DC from being doubled

    # plot
    plt.figure(figsize=(8, 4))
    plt.plot(freq, amp)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Amplitude")
    plt.title(f"Amplitude spectrum: {tr.id}")
    plt.xlim(0, tr.stats.sampling_rate / 2)
    plt.grid(True)
    plt.show()

for tr in master_stream_zoom.copy():
    plot_spectrum(tr)

In [ ]:
import numpy as np
from obspy import read
from scipy.io.wavfile import write as wavwrite
from scipy.signal import resample

def trace_to_audio(
    tr,
    speedup=100,
    output_wav="output.wav",
    audio_rate=44100,
    normalize=True,
    detrend=True,
    taper=True,
    bandpass=None,
):
    """
    Convert an ObsPy Trace to a WAV audio file by speeding it up.

    Parameters
    ----------
    tr : obspy.Trace
        Input seismic trace.
    speedup : float
        Factor by which to speed up playback.
    output_wav : str
        Output WAV filename.
    audio_rate : int
        Desired WAV sample rate in Hz.
    normalize : bool
        If True, scale data to int16 range.
    detrend : bool
        If True, remove mean and linear trend.
    taper : bool
        If True, apply a short taper.
    bandpass : tuple or None
        Optional (fmin, fmax) in Hz for pre-filtering the seismic data.
    """
    tr = tr.copy()

    if detrend:
        tr.detrend("demean")
        tr.detrend("linear")

    if taper:
        tr.taper(max_percentage=0.02)

    if bandpass is not None:
        fmin, fmax = bandpass
        tr.filter("bandpass", freqmin=fmin, freqmax=fmax, corners=4, zerophase=True)

    data = tr.data.astype(np.float64)

    # Remove NaNs/infs if present
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

    # Effective sampling rate after speeding up
    effective_rate = tr.stats.sampling_rate * speedup

    # Resample to desired audio rate if needed
    if abs(effective_rate - audio_rate) / audio_rate > 0.01:
        n_out = int(round(len(data) * audio_rate / effective_rate))
        if n_out <= 0:
            raise ValueError("Output sample count is invalid.")
        data = resample(data, n_out)

    if normalize:
        peak = np.max(np.abs(data))
        if peak > 0:
            data = data / peak
        data_int16 = np.int16(data * 32767)
    else:
        data_int16 = np.int16(np.clip(data, -32768, 32767))

    wavwrite(output_wav, audio_rate, data_int16)
    return output_wav

In [ ]:
for tr in master_stream_zoom.copy():
    output_filename = f"{tr.id.replace('.', '_')}_audio.wav"
    trace_to_audio(tr, speedup=100, output_wav=output_filename, bandpass=(0.1, 48))  